# HEALTHCARE PROVIDER FRAUD DETECTION ANALYSIS

**Problem Statement**
The goal of this project is to " predict the potentially fraudulent providers " based on the claims filed by them.along with this, we will also discover important variables helpful in detecting the behaviour of potentially fraud providers. further, we will study fraudulent patterns in the provider's claims to understand the future behaviour of providers.

A) Inpatient Data

This data provides insights about the claims filed for those patients who are admitted in the hospitals. It also provides additional details like their admission and discharge dates and admit d diagnosis code.

B) Outpatient Data

This data provides details about the claims filed for those patients who visit hospitals and not admitted in it.

C) Beneficiary Details Data

This data contains beneficiary KYC details like health conditions,regioregion they belong to etc

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
from matplotlib import pyplot

import warnings
warnings.filterwarnings("ignore")

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Beneficiary 
This data contains beneficiary KYC details like health conditions,regioregion they belong to etc

## Data Preprocessing and EDA

In [ ]:
df_train_ben = pd.read_csv("/kaggle/input/healthcare-provider-fraud-detection-analysis/Train_Beneficiarydata-1542865627584.csv")
df_train_ben = df_train_ben.drop_duplicates()
df_train_ben.head()

In [ ]:
df_train_ben.columns

In [ ]:
df_train_ben.dtypes

In total there are 138556 beneficiaries.

In [ ]:
df_train_ben.shape

**Null Values**

In the Beneficiary dataset, DOD is the only column that mostly filled with NaN values.

In [ ]:
sns.heatmap(df_train_ben.isnull(),yticklabels=False,cbar=False,cmap='viridis')

In [ ]:
df_train_ben['DOD'] = pd.to_datetime(df_train_ben['DOD'], format='%Y-%m-%d')
df_train_ben['DOB'] = pd.to_datetime(df_train_ben['DOB'], format='%Y-%m-%d')

In [ ]:
df_train_ben["DOD"].max()

In [ ]:
df_train_ben[df_train_ben.DOD == df_train_ben.DOD.max()]

The most recent DOD value is December 1, 2009, which indicates that the Beneficiary Details data is from 2009.

Therefore NaN values in the DOD columns are filled with '2009-12-01' 

In [ ]:
df_train_ben["DOD"].fillna("2009-12-01", inplace = True)

**Adding Age Column**

In [ ]:
df_train_ben['Age'] = df_train_ben['DOD'].dt.year - df_train_ben['DOB'].dt.year

In [ ]:
import matplotlib.pyplot as plt
df1=df_train_ben.select_dtypes(include=['int64'])
for column in df1:
        plt.figure(figsize=(17,1))
        sns.boxplot(data=df1, x=column)

In [ ]:
def with_hue(ax, feature, Number_of_categories, hue_categories):
    a = [p.get_height() for p in ax.patches]
    patch = [p for p in ax.patches]
    for i in range(Number_of_categories):
        total = feature.value_counts().values[i]
        for j in range(hue_categories):
            percentage = '{:.1f}%'.format(100 * a[(j*Number_of_categories + i)]/total)
            x = patch[(j*Number_of_categories + i)].get_x() + patch[(j*Number_of_categories + i)].get_width() / 2 - 0.15
            y = patch[(j*Number_of_categories + i)].get_y() + patch[(j*Number_of_categories + i)].get_height() 
            ax.annotate(percentage, (x, y), size = 12)
            
def without_hue(plot, feature):
    total = len(feature)
    for p in ax.patches:
        percentage = '{:.1f}%'.format(100 * p.get_height()/total)
        x = p.get_x() + p.get_width() / 2 - 0.05
        y = p.get_y() + p.get_height()
        ax.annotate(percentage, (x, y), size = 12)
    plt.show()

**Gender**

The gender column has a roughly even distribution of data, however the second gender accounts for 15% more entries.

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('Gender', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('Gender', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.Gender)

**Race**

Race column has a data spread dominated by 1. 

It indicates that they are the majority of the Beneficiaries.

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('Race', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('Race', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.Race)

**Age**

In [ ]:
plt.figure(figsize=(30,20))
ax = sns.countplot('Age', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('Age', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.Age)

In [ ]:
conditions = [
    (df_train_ben['Age'] <= 34),
    (df_train_ben['Age'] > 34) & (df_train_ben['Age'] <= 44),
    (df_train_ben['Age'] >= 45) & (df_train_ben['Age'] <= 64),
    (df_train_ben['Age'] >= 65)
    ]

values = ['Early Adulthood','Early Middle Age','Late Middle Age' ,'Late Adulthood']
df_train_ben['Age'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Age', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('Age', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.Age)

**State**

In [ ]:
plt.figure(figsize=(20,20))
ax = sns.countplot('State', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('State', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.State)

**County**

In [ ]:
plt.figure(figsize=(20,20))
ax = sns.countplot('County', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('County', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.County)

**RenalDiseaseIndicator**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('RenalDiseaseIndicator', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('RenalDiseaseIndicator', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.RenalDiseaseIndicator)

**NoOfMonths_PartACov**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('NoOfMonths_PartACov', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('NoOfMonths_PartACov', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.NoOfMonths_PartACov)

In [ ]:
df_train_ben = df_train_ben.drop('NoOfMonths_PartACov', axis=1)

**NoOfMonths_PartBCov**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('NoOfMonths_PartBCov', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('NoOfMonths_PartBCov', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.NoOfMonths_PartBCov)

In [ ]:
df_train_ben = df_train_ben.drop('NoOfMonths_PartBCov', axis=1)

**ChronicCond_Alzheimer**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('ChronicCond_Alzheimer', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('ChronicCond_Alzheimer', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.ChronicCond_Alzheimer)

**ChronicCond_Heartfailure**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('ChronicCond_Heartfailure', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('ChronicCond_Heartfailure', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.ChronicCond_Heartfailure)

**ChronicCond_KidneyDisease**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('ChronicCond_KidneyDisease', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('ChronicCond_KidneyDisease', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.ChronicCond_KidneyDisease)

**ChronicCond_Cancer**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('ChronicCond_Cancer', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('ChronicCond_Cancer', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.ChronicCond_Cancer)

**ChronicCond_ObstrPulmonary**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('ChronicCond_ObstrPulmonary', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('ChronicCond_ObstrPulmonary', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.ChronicCond_ObstrPulmonary)

**ChronicCond_Depression**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('ChronicCond_Depression', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('ChronicCond_Depression', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.ChronicCond_Depression)

**ChronicCond_Diabetes**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('ChronicCond_Diabetes', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('ChronicCond_Diabetes', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.ChronicCond_Depression)

**ChronicCond_IschemicHeart**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('ChronicCond_IschemicHeart', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('ChronicCond_IschemicHeart', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.ChronicCond_IschemicHeart)

**ChronicCond_Osteoporasis**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('ChronicCond_Osteoporasis', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('ChronicCond_Osteoporasis', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.ChronicCond_Osteoporasis)

**ChronicCond_rheumatoidarthritis**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('ChronicCond_rheumatoidarthritis', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('ChronicCond_rheumatoidarthritis', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.ChronicCond_rheumatoidarthritis)

**ChronicCond_stroke**

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('ChronicCond_stroke', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('ChronicCond_stroke', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.ChronicCond_stroke)

# Provider Data
Healthcare fraud is an organized crime which involves peers of providers, physicians, beneficiaries acting together to make fraud claims. Therefore, focusing first on the Provider would be the best option for now.

In [ ]:
df_train_prov = pd.read_csv("/kaggle/input/healthcare-provider-fraud-detection-analysis/Train-1542865627584.csv")
df_train_prov.head()

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_prov)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_prov.PotentialFraud)

In [ ]:
df_fraud_prov = df_train_prov[df_train_prov.PotentialFraud == 'Yes']
df_fraud_prov.head()

There are 506 potential fraud providers.

In [ ]:
df_fraud_prov.shape

# Inpatient Data

In [ ]:
df_train_ip = pd.read_csv("/kaggle/input/healthcare-provider-fraud-detection-analysis/Train_Inpatientdata-1542865627584.csv")
df_train_ip.head()

In [ ]:
df_train_ip.columns

In [ ]:
df_train_ip.dtypes

In [ ]:
df_train_ip.info()

In [ ]:
df_train_ip.describe()

In [ ]:
df_train_ip.shape

In [ ]:
import matplotlib.pyplot as plt
df1=df_train_ip.select_dtypes(include=['int64'])
for column in df1:
        plt.figure(figsize=(17,1))
        sns.boxplot(data=df1, x=column)

In [ ]:
sns.heatmap(df_train_ip.isnull(),yticklabels=False,cbar=False,cmap='viridis')

In [ ]:
df_train_ip['AttendingPhysician'] = df_train_ip['AttendingPhysician'].fillna("None")
df_train_ip['OperatingPhysician'] = df_train_ip['OperatingPhysician'].fillna("None")
df_train_ip['OtherPhysician'] = df_train_ip['OtherPhysician'].fillna("None")

In [ ]:
df_train_ip = df_train_ip.fillna("0")

In [ ]:
df_train_ip['BeneID'].nunique()

In [ ]:
df_train_ip['BeneID'].value_counts()

In [ ]:
df_train_ben["has_claimed_ip"] = df_train_ben["BeneID"].isin(df_train_ip["BeneID"])

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('has_claimed_ip', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('has_claimed_ip', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.has_claimed_ip)

**Adding Claim Duration**

In [ ]:
df_train_ip['ClaimEndDt'] = pd.to_datetime(df_train_ip['ClaimEndDt'], format='%Y-%m-%d')
df_train_ip['ClaimStartDt'] = pd.to_datetime(df_train_ip['ClaimStartDt'], format='%Y-%m-%d')

In [ ]:
df_train_ip["claim_duration"] = (df_train_ip['ClaimEndDt'] - df_train_ip['ClaimStartDt']).dt.days
df_train_ip.head()

**Merging IP data to the Provider data**

In [ ]:
df_train_ip = df_train_ip.join(df_train_prov.set_index('Provider'), on='Provider')

In [ ]:
df_train_ip.columns

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_ip)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ip.PotentialFraud)

In [ ]:
df_train_ip['AttendingPhysician'].nunique()

In [ ]:
df_train_ip['OperatingPhysician'].nunique()

In [ ]:
df_train_ip.shape

**Adding Beneficiaries data to Claim data**

In [ ]:
df_train_ip_total = df_train_ip.join(df_train_ben.set_index('BeneID'), on='BeneID')
df_train_ip_total.head()

In [ ]:
df_train_ip_total['ChronicCond_Alzheimer'].unique()

In [ ]:
sns.heatmap(df_train_ip_total.isnull(),yticklabels=False,cbar=False,cmap='viridis')

In [ ]:
df_train_ip_total

## Diseases

### ChronicCond_Alzheimer

In [ ]:
# df_train_ip_Alz = df_train_ip_total[(df_train_ip_total['ChronicCond_Alzheimer'] == 1) & (df_train_ip_total['ChronicCond_Heartfailure'] == 2) & (df_train_ip_total['ChronicCond_KidneyDisease'] == 2) & (df_train_ip_total['ChronicCond_Cancer'] == 2) & (df_train_ip_total['ChronicCond_ObstrPulmonary'] == 2) & (df_train_ip_total['ChronicCond_Depression'] == 2) & (df_train_ip_total['ChronicCond_Diabetes'] == 2) & (df_train_ip_total['ChronicCond_IschemicHeart'] == 2) & (df_train_ip_total['ChronicCond_Osteoporasis'] == 2) & (df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 2)& (df_train_ip_total['ChronicCond_stroke'] == 2)]
df_train_ip_Alz = df_train_ip_total[df_train_ip_total['ChronicCond_Alzheimer'] == 1]
df_train_ip_Alz.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_ip_Alz)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ip_Alz.PotentialFraud)

Let's see the average cost of inpatient Alzheimer Patient

In [ ]:
df_train_ip_Alz_not_fraud = df_train_ip_Alz[df_train_ip_Alz['PotentialFraud'] == "No"]

In [ ]:
df_train_ip_Alz_not_fraud.shape

In [ ]:
df_train_ip_Alz_not_fraud.head()

In [ ]:
df_train_ip_Alz_not_fraud.dtypes

In [ ]:
df_train_ip_Alz_not_fraud.describe()

In [ ]:
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

In [ ]:
conditions = [
    (df_train_ip_Alz['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_ip_Alz['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_ip_Alz['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_ip_Alz['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_ip_Alz['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_ip_Alz['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_ip_Alz['Alz_claim_ip_insurance_tier'] = np.select(conditions, values)

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('Alz_claim_ip_insurance_tier', hue= 'PotentialFraud', data = df_train_ip_Alz)
plt.xticks(size = 12)
plt.xlabel('Alz_claim_ip_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_ip_Alz.Alz_claim_ip_insurance_tier,2,2)

In [ ]:
df_train_ip_Alz.head()

In [ ]:
df_train_ip_Alz.dtypes

In [ ]:
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_ip_Alz['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_ip_Alz['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_ip_Alz['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_ip_Alz['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_ip_Alz['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_ip_Alz['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_ip_Alz['Alz_claim_ip_insurance_tier'] = np.select(conditions, values)

In [ ]:
df_train_ip_total['Alz_claim_ip_insurance_tier'] = df_train_ip_Alz['Alz_claim_ip_insurance_tier']
df_train_ip_total["Alz_claim_ip_insurance_tier"].fillna("None", inplace = True)
df_train_ip_total.head()

### Heartfailure

In [ ]:
# df_train_ip_Alz = df_train_ip_total[(df_train_ip_total['ChronicCond_Alzheimer'] == 1) & (df_train_ip_total['ChronicCond_Heartfailure'] == 2) & (df_train_ip_total['ChronicCond_KidneyDisease'] == 2) & (df_train_ip_total['ChronicCond_Cancer'] == 2) & (df_train_ip_total['ChronicCond_ObstrPulmonary'] == 2) & (df_train_ip_total['ChronicCond_Depression'] == 2) & (df_train_ip_total['ChronicCond_Diabetes'] == 2) & (df_train_ip_total['ChronicCond_IschemicHeart'] == 2) & (df_train_ip_total['ChronicCond_Osteoporasis'] == 2) & (df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 2)& (df_train_ip_total['ChronicCond_stroke'] == 2)]
df_train_ip_Hf = df_train_ip_total[df_train_ip_total['ChronicCond_Heartfailure'] == 1]
df_train_ip_Hf.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_ip_Hf)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ip_Hf.PotentialFraud)

In [ ]:
df_train_ip_Hf_not_fraud = df_train_ip_Hf[df_train_ip_Hf['PotentialFraud'] == "No"]

In [ ]:
df_train_ip_Hf_not_fraud.shape

In [ ]:
df_train_ip_Hf_not_fraud.head()

In [ ]:
df_train_ip_Hf_not_fraud.dtypes

In [ ]:
df_train_ip_Hf_not_fraud.describe()

In [ ]:
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_ip_Hf_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

In [ ]:
conditions = [
    (df_train_ip_Hf['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_ip_Hf['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_ip_Hf['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_ip_Hf['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_ip_Hf['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_ip_Hf['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_ip_Hf['Hf_claim_ip_insurance_tier'] = np.select(conditions, values)

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('Hf_claim_ip_insurance_tier', hue= 'PotentialFraud', data = df_train_ip_Hf)
plt.xticks(size = 12)
plt.xlabel('Hf_claim_ip_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_ip_Hf.Hf_claim_ip_insurance_tier,2,2)

In [ ]:
df_train_ip_total['Hf_claim_ip_insurance_tier'] = df_train_ip_Hf['Hf_claim_ip_insurance_tier']
df_train_ip_total["Hf_claim_ip_insurance_tier"].fillna("None", inplace = True)
df_train_ip_total.head()

### KidneyDisease

In [ ]:
# df_train_ip_Alz = df_train_ip_total[(df_train_ip_total['ChronicCond_Alzheimer'] == 1) & (df_train_ip_total['ChronicCond_Heartfailure'] == 2) & (df_train_ip_total['ChronicCond_KidneyDisease'] == 2) & (df_train_ip_total['ChronicCond_Cancer'] == 2) & (df_train_ip_total['ChronicCond_ObstrPulmonary'] == 2) & (df_train_ip_total['ChronicCond_Depression'] == 2) & (df_train_ip_total['ChronicCond_Diabetes'] == 2) & (df_train_ip_total['ChronicCond_IschemicHeart'] == 2) & (df_train_ip_total['ChronicCond_Osteoporasis'] == 2) & (df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 2)& (df_train_ip_total['ChronicCond_stroke'] == 2)]
df_train_ip_Kd = df_train_ip_total[df_train_ip_total['ChronicCond_KidneyDisease'] == 1]
df_train_ip_Kd.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_ip_Kd)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ip_Kd.PotentialFraud)

In [ ]:
df_train_ip_Kd_not_fraud = df_train_ip_Kd[df_train_ip_Kd['PotentialFraud'] == "No"]

In [ ]:
df_train_ip_Kd_not_fraud.shape

In [ ]:
df_train_ip_Kd_not_fraud.head()

In [ ]:
df_train_ip_Kd_not_fraud.dtypes

In [ ]:
df_train_ip_Kd_not_fraud.describe()

In [ ]:
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_ip_Kd_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

In [ ]:
conditions = [
    (df_train_ip_Kd['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_ip_Kd['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_ip_Kd['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_ip_Kd['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_ip_Kd['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_ip_Kd['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_ip_Kd['Kd_claim_insurance_tier'] = np.select(conditions, values)

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('Kd_claim_insurance_tier', hue= 'PotentialFraud', data = df_train_ip_Kd)
plt.xticks(size = 12)
plt.xlabel('Kd_claim_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_ip_Kd.Kd_claim_insurance_tier,2,2)

In [ ]:
df_train_ip_total['Kd_claim_insurance_tier'] = df_train_ip_Kd['Kd_claim_insurance_tier']
df_train_ip_total["Kd_claim_insurance_tier"].fillna("None", inplace = True)
df_train_ip_total.head()

### Cancer

In [ ]:
# df_train_ip_Alz = df_train_ip_total[(df_train_ip_total['ChronicCond_Alzheimer'] == 1) & (df_train_ip_total['ChronicCond_Heartfailure'] == 2) & (df_train_ip_total['ChronicCond_KidneyDisease'] == 2) & (df_train_ip_total['ChronicCond_Cancer'] == 2) & (df_train_ip_total['ChronicCond_ObstrPulmonary'] == 2) & (df_train_ip_total['ChronicCond_Depression'] == 2) & (df_train_ip_total['ChronicCond_Diabetes'] == 2) & (df_train_ip_total['ChronicCond_IschemicHeart'] == 2) & (df_train_ip_total['ChronicCond_Osteoporasis'] == 2) & (df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 2)& (df_train_ip_total['ChronicCond_stroke'] == 2)]
df_train_ip_Ca = df_train_ip_total[df_train_ip_total['ChronicCond_Cancer'] == 1]
df_train_ip_Ca.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_ip_Ca)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ip_Ca.PotentialFraud)

In [ ]:
df_train_ip_Ca_not_fraud = df_train_ip_Ca[df_train_ip_Ca['PotentialFraud'] == "No"]

In [ ]:
df_train_ip_Ca_not_fraud.shape

In [ ]:
df_train_ip_Ca_not_fraud.head()

In [ ]:
df_train_ip_Ca_not_fraud.dtypes

In [ ]:
df_train_ip_Ca_not_fraud.describe()

In [ ]:
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_ip_Ca_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_ip_Ca['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_ip_Ca['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_ip_Ca['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_ip_Ca['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_ip_Ca['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_ip_Ca['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_ip_Ca['Ca_claim_ip_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Ca_claim_ip_insurance_tier', hue= 'PotentialFraud', data = df_train_ip_Ca)
plt.xticks(size = 12)
plt.xlabel('Ca_claim_ip_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_ip_Ca.Ca_claim_ip_insurance_tier,2,2)

In [ ]:
df_train_ip_total['Ca_claim_ip_insurance_tier'] = df_train_ip_Ca['Ca_claim_ip_insurance_tier']
df_train_ip_total["Ca_claim_ip_insurance_tier"].fillna("None", inplace = True)
df_train_ip_total.head()

### ObstrPulmonary

In [ ]:
# df_train_ip_Alz = df_train_ip_total[(df_train_ip_total['ChronicCond_Alzheimer'] == 1) & (df_train_ip_total['ChronicCond_Heartfailure'] == 2) & (df_train_ip_total['ChronicCond_KidneyDisease'] == 2) & (df_train_ip_total['ChronicCond_Cancer'] == 2) & (df_train_ip_total['ChronicCond_ObstrPulmonary'] == 2) & (df_train_ip_total['ChronicCond_Depression'] == 2) & (df_train_ip_total['ChronicCond_Diabetes'] == 2) & (df_train_ip_total['ChronicCond_IschemicHeart'] == 2) & (df_train_ip_total['ChronicCond_Osteoporasis'] == 2) & (df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 2)& (df_train_ip_total['ChronicCond_stroke'] == 2)]
df_train_ip_Obs = df_train_ip_total[df_train_ip_total['ChronicCond_ObstrPulmonary'] == 1]
df_train_ip_Obs.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_ip_Obs)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ip_Obs.PotentialFraud)

In [ ]:
df_train_ip_Obs_not_fraud = df_train_ip_Obs[df_train_ip_Obs['PotentialFraud'] == "No"]

In [ ]:
df_train_ip_Obs_not_fraud.shape

In [ ]:
df_train_ip_Obs_not_fraud.head()

In [ ]:
df_train_ip_Obs_not_fraud.dtypes

In [ ]:
df_train_ip_Obs_not_fraud.describe()

In [ ]:
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_ip_Obs_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_ip_Obs['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_ip_Obs['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_ip_Obs['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_ip_Obs['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_ip_Obs['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_ip_Obs['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_ip_Obs['Obs_claim_ip_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Obs_claim_ip_insurance_tier', hue= 'PotentialFraud', data = df_train_ip_Obs)
plt.xticks(size = 12)
plt.xlabel('Obs_claim_ip_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_ip_Obs.Obs_claim_ip_insurance_tier,2,2)

In [ ]:
df_train_ip_total['Obs_claim_ip_insurance_tier'] = df_train_ip_Obs['Obs_claim_ip_insurance_tier']
df_train_ip_total["Obs_claim_ip_insurance_tier"].fillna("None", inplace = True)
df_train_ip_total.head()

### Depression

In [ ]:
# df_train_ip_Alz = df_train_ip_total[(df_train_ip_total['ChronicCond_Alzheimer'] == 1) & (df_train_ip_total['ChronicCond_Heartfailure'] == 2) & (df_train_ip_total['ChronicCond_KidneyDisease'] == 2) & (df_train_ip_total['ChronicCond_Cancer'] == 2) & (df_train_ip_total['ChronicCond_ObstrPulmonary'] == 2) & (df_train_ip_total['ChronicCond_Depression'] == 2) & (df_train_ip_total['ChronicCond_Diabetes'] == 2) & (df_train_ip_total['ChronicCond_IschemicHeart'] == 2) & (df_train_ip_total['ChronicCond_Osteoporasis'] == 2) & (df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 2)& (df_train_ip_total['ChronicCond_stroke'] == 2)]
df_train_ip_Dep = df_train_ip_total[df_train_ip_total['ChronicCond_Depression'] == 1]
df_train_ip_Dep.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_ip_Dep)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ip_Dep.PotentialFraud)

In [ ]:
df_train_ip_Dep_not_fraud = df_train_ip_Dep[df_train_ip_Dep['PotentialFraud'] == "No"]

In [ ]:
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_ip_Dep_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_ip_Dep['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_ip_Dep['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_ip_Dep['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_ip_Dep['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_ip_Dep['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_ip_Dep['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_ip_Dep['Dep_claim_ip_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Dep_claim_ip_insurance_tier', hue= 'PotentialFraud', data = df_train_ip_Dep)
plt.xticks(size = 12)
plt.xlabel('Dep_claim_ip_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_ip_Dep.Dep_claim_ip_insurance_tier,2,2)

In [ ]:
df_train_ip_total['Dep_claim_ip_insurance_tier'] = df_train_ip_Dep['Dep_claim_ip_insurance_tier']
df_train_ip_total["Dep_claim_ip_insurance_tier"].fillna("None", inplace = True)
df_train_ip_total.head()

### Diabetes

In [ ]:
# df_train_ip_Alz = df_train_ip_total[(df_train_ip_total['ChronicCond_Alzheimer'] == 1) & (df_train_ip_total['ChronicCond_Heartfailure'] == 2) & (df_train_ip_total['ChronicCond_KidneyDisease'] == 2) & (df_train_ip_total['ChronicCond_Cancer'] == 2) & (df_train_ip_total['ChronicCond_ObstrPulmonary'] == 2) & (df_train_ip_total['ChronicCond_Depression'] == 2) & (df_train_ip_total['ChronicCond_Diabetes'] == 2) & (df_train_ip_total['ChronicCond_IschemicHeart'] == 2) & (df_train_ip_total['ChronicCond_Osteoporasis'] == 2) & (df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 2)& (df_train_ip_total['ChronicCond_stroke'] == 2)]
df_train_ip_Dia = df_train_ip_total[df_train_ip_total['ChronicCond_Diabetes'] == 1]
df_train_ip_Dia.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_ip_Dia)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ip_Dia.PotentialFraud)

In [ ]:
df_train_ip_Dia_not_fraud = df_train_ip_Dia[df_train_ip_Dia['PotentialFraud'] == "No"]

# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_ip_Dia_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_ip_Dia['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_ip_Dia['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_ip_Dia['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_ip_Dia['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_ip_Dia['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_ip_Dia['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_ip_Dia['Dia_claim_ip_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Dia_claim_ip_insurance_tier', hue= 'PotentialFraud', data = df_train_ip_Dia)
plt.xticks(size = 12)
plt.xlabel('Dia_claim_ip_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_ip_Dia.Dia_claim_ip_insurance_tier,2,2)

In [ ]:
df_train_ip_total['Dia_claim_ip_insurance_tier'] = df_train_ip_Dia['Dia_claim_ip_insurance_tier']
df_train_ip_total["Dia_claim_ip_insurance_tier"].fillna("None", inplace = True)
df_train_ip_total.head()

### Ischemic Heart

In [ ]:
# df_train_ip_Alz = df_train_ip_total[(df_train_ip_total['ChronicCond_Alzheimer'] == 1) & (df_train_ip_total['ChronicCond_Heartfailure'] == 2) & (df_train_ip_total['ChronicCond_KidneyDisease'] == 2) & (df_train_ip_total['ChronicCond_Cancer'] == 2) & (df_train_ip_total['ChronicCond_ObstrPulmonary'] == 2) & (df_train_ip_total['ChronicCond_Depression'] == 2) & (df_train_ip_total['ChronicCond_Diabetes'] == 2) & (df_train_ip_total['ChronicCond_IschemicHeart'] == 2) & (df_train_ip_total['ChronicCond_Osteoporasis'] == 2) & (df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 2)& (df_train_ip_total['ChronicCond_stroke'] == 2)]
df_train_ip_Isc = df_train_ip_total[df_train_ip_total['ChronicCond_IschemicHeart'] == 1]
df_train_ip_Isc.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_ip_Isc)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ip_Isc.PotentialFraud)

In [ ]:
df_train_ip_Isc_not_fraud = df_train_ip_Isc[df_train_ip_Isc['PotentialFraud'] == "No"]
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_ip_Isc_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_ip_Isc['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_ip_Isc['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_ip_Isc['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_ip_Isc['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_ip_Isc['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_ip_Isc['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_ip_Isc['Isc_claim_ip_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Isc_claim_ip_insurance_tier', hue= 'PotentialFraud', data = df_train_ip_Isc)
plt.xticks(size = 12)
plt.xlabel('Isc_claim_ip_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_ip_Isc.Isc_claim_ip_insurance_tier,2,2)

In [ ]:
df_train_ip_total['Isc_claim_ip_insurance_tier'] = df_train_ip_Isc['Isc_claim_ip_insurance_tier']
df_train_ip_total["Isc_claim_ip_insurance_tier"].fillna("None", inplace = True)
df_train_ip_total.head()

### Osteoporasis

In [ ]:
# df_train_ip_Alz = df_train_ip_total[(df_train_ip_total['ChronicCond_Alzheimer'] == 1) & (df_train_ip_total['ChronicCond_Heartfailure'] == 2) & (df_train_ip_total['ChronicCond_KidneyDisease'] == 2) & (df_train_ip_total['ChronicCond_Cancer'] == 2) & (df_train_ip_total['ChronicCond_ObstrPulmonary'] == 2) & (df_train_ip_total['ChronicCond_Depression'] == 2) & (df_train_ip_total['ChronicCond_Diabetes'] == 2) & (df_train_ip_total['ChronicCond_IschemicHeart'] == 2) & (df_train_ip_total['ChronicCond_Osteoporasis'] == 2) & (df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 2)& (df_train_ip_total['ChronicCond_stroke'] == 2)]
df_train_ip_Ost = df_train_ip_total[df_train_ip_total['ChronicCond_Osteoporasis'] == 1]
df_train_ip_Ost.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_ip_Ost)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ip_Ost.PotentialFraud)

In [ ]:
df_train_ip_Ost_not_fraud = df_train_ip_Ost[df_train_ip_Ost['PotentialFraud'] == "No"]
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_ip_Ost_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_ip_Ost['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_ip_Ost['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_ip_Ost['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_ip_Ost['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_ip_Ost['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_ip_Ost['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_ip_Ost['Ost_claim_ip_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Ost_claim_ip_insurance_tier', hue= 'PotentialFraud', data = df_train_ip_Ost)
plt.xticks(size = 12)
plt.xlabel('Ost_claim_ip_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_ip_Ost.Ost_claim_ip_insurance_tier,2,2)

In [ ]:
df_train_ip_total['Ost_claim_ip_insurance_tier'] = df_train_ip_Ost['Ost_claim_ip_insurance_tier']
df_train_ip_total["Ost_claim_ip_insurance_tier"].fillna("None", inplace = True)
df_train_ip_total.head()

### Rheumatoidarthritis

In [ ]:
# df_train_ip_Alz = df_train_ip_total[(df_train_ip_total['ChronicCond_Alzheimer'] == 1) & (df_train_ip_total['ChronicCond_Heartfailure'] == 2) & (df_train_ip_total['ChronicCond_KidneyDisease'] == 2) & (df_train_ip_total['ChronicCond_Cancer'] == 2) & (df_train_ip_total['ChronicCond_ObstrPulmonary'] == 2) & (df_train_ip_total['ChronicCond_Depression'] == 2) & (df_train_ip_total['ChronicCond_Diabetes'] == 2) & (df_train_ip_total['ChronicCond_IschemicHeart'] == 2) & (df_train_ip_total['ChronicCond_Osteoporasis'] == 2) & (df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 2)& (df_train_ip_total['ChronicCond_stroke'] == 2)]
df_train_ip_Rhe = df_train_ip_total[df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 1]
df_train_ip_Rhe.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_ip_Rhe)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ip_Rhe.PotentialFraud)

In [ ]:
df_train_ip_Rhe_not_fraud = df_train_ip_Rhe[df_train_ip_Rhe['PotentialFraud'] == "No"]
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_ip_Rhe_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_ip_Rhe['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_ip_Rhe['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_ip_Rhe['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_ip_Rhe['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_ip_Rhe['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_ip_Rhe['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['very_expensive', 'expensive', 'regular', 'inexpensive']
df_train_ip_Rhe['Rhe_claim_ip_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Rhe_claim_ip_insurance_tier', hue= 'PotentialFraud', data = df_train_ip_Rhe)
plt.xticks(size = 12)
plt.xlabel('Rhe_claim_ip_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_ip_Rhe.Rhe_claim_ip_insurance_tier,2,2)

In [ ]:
df_train_ip_total['Rhe_claim_ip_insurance_tier'] = df_train_ip_Rhe['Rhe_claim_ip_insurance_tier']
df_train_ip_total["Rhe_claim_ip_insurance_tier"].fillna("None", inplace = True)
df_train_ip_total.head()

### Stroke

In [ ]:
# df_train_ip_Alz = df_train_ip_total[(df_train_ip_total['ChronicCond_Alzheimer'] == 1) & (df_train_ip_total['ChronicCond_Heartfailure'] == 2) & (df_train_ip_total['ChronicCond_KidneyDisease'] == 2) & (df_train_ip_total['ChronicCond_Cancer'] == 2) & (df_train_ip_total['ChronicCond_ObstrPulmonary'] == 2) & (df_train_ip_total['ChronicCond_Depression'] == 2) & (df_train_ip_total['ChronicCond_Diabetes'] == 2) & (df_train_ip_total['ChronicCond_IschemicHeart'] == 2) & (df_train_ip_total['ChronicCond_Osteoporasis'] == 2) & (df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 2)& (df_train_ip_total['ChronicCond_stroke'] == 2)]
df_train_ip_Str = df_train_ip_total[df_train_ip_total['ChronicCond_Depression'] == 1]
df_train_ip_Str.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_ip_Str)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ip_Str.PotentialFraud)

In [ ]:
df_train_ip_Str_not_fraud = df_train_ip_Str[df_train_ip_Str['PotentialFraud'] == "No"]
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_ip_Str_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_ip_Str['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_ip_Str['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_ip_Str['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_ip_Str['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_ip_Str['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_ip_Str['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_ip_Str['Str_claim_ip_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Str_claim_ip_insurance_tier', hue= 'PotentialFraud', data = df_train_ip_Str)
plt.xticks(size = 12)
plt.xlabel('Str_claim_ip_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_ip_Str.Str_claim_ip_insurance_tier,2,2)

In [ ]:
df_train_ip_total['Str_claim_ip_insurance_tier'] = df_train_ip_Str['Str_claim_ip_insurance_tier']
df_train_ip_total["Str_claim_ip_insurance_tier"].fillna("None", inplace = True)
df_train_ip_total.head()

### Outpatient Data

In [ ]:
df_train_op = pd.read_csv("/kaggle/input/healthcare-provider-fraud-detection-analysis/Train_Outpatientdata-1542865627584.csv")
df_train_op.head()

In [ ]:
df_train_op.columns

In [ ]:
df_train_op.dtypes

In [ ]:
df_train_op.info()

In [ ]:
df_train_op.describe()

In [ ]:
df_train_op.shape

In [ ]:
df_train_op['AttendingPhysician'] = df_train_op['AttendingPhysician'].fillna("None")
df_train_op['OperatingPhysician'] = df_train_op['OperatingPhysician'].fillna("None")
df_train_op['OtherPhysician'] = df_train_op['OtherPhysician'].fillna("None")

df_train_op=df_train_op.fillna(0)

In [ ]:
df_train_op['BeneID'].nunique()

**Benerficiaries Claims**

We can see several Benerficiaries have claimed several times

In [ ]:
df_train_op['BeneID'].value_counts()

In [ ]:
df_train_ben["has_claimed_op"] = df_train_ben["BeneID"].isin(df_train_op["BeneID"])

plt.figure(figsize=(7,5))
ax = sns.countplot('has_claimed_op', data = df_train_ben)
plt.xticks(size = 12)
plt.xlabel('has_claimed_op', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_ben.has_claimed_op)

About 96.7% percent of beneficiaries have admitted OP claimed

**Adding Claim Duration**

In [ ]:
df_train_op['ClaimEndDt'] = pd.to_datetime(df_train_op['ClaimEndDt'], format='%Y-%m-%d')
df_train_op['ClaimStartDt'] = pd.to_datetime(df_train_op['ClaimStartDt'], format='%Y-%m-%d')

df_train_op["claim_duration"] = df_train_op['ClaimEndDt'] - df_train_op['ClaimStartDt']
df_train_op.head()

**Merging OP data with Provider Data**

In [ ]:
df_train_op_total = df_train_op.join(df_train_prov.set_index('Provider'), on='Provider')
df_train_op_total.head()

In [ ]:
df_train_op_total.dtypes

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_op_total)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_op_total.PotentialFraud)

**Adding Beneficiaries data to Claim data**

In [ ]:
df_train_op_total = df_train_op_total.join(df_train_ben.set_index('BeneID'), on='BeneID')
df_train_op_total.head()

## Disease

### Alzheimer

In [ ]:
df_train_op_Alz = df_train_op_total[df_train_op_total['ChronicCond_Alzheimer'] == 1]
df_train_op_Alz.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_op_Alz)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_op_Alz.PotentialFraud)

In [ ]:
df_train_op_Alz_not_fraud = df_train_op_Alz[df_train_op_Alz['PotentialFraud'] == "No"]
lower_quantile, middle_quantile,upper_quantile = df_train_op_Alz_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])
conditions = [
    (df_train_op_Alz['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_op_Alz['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_op_Alz['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_op_Alz['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_op_Alz['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_op_Alz['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_op_Alz['Alz_claim_op_insurance_tier'] = np.select(conditions, values)

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('Alz_claim_op_insurance_tier', hue= 'PotentialFraud', data = df_train_op_Alz)
plt.xticks(size = 12)
plt.xlabel('Alz_claim_op_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_op_Alz.Alz_claim_op_insurance_tier,2,2)

In [ ]:
df_train_op_total['Alz_claim_op_insurance_tier'] = df_train_op_Alz['Alz_claim_op_insurance_tier']
df_train_op_total["Alz_claim_op_insurance_tier"].fillna("None", inplace = True)
df_train_op_total.head()

### Heartfailure

In [ ]:
df_train_op_Hf = df_train_op_total[df_train_op_total['ChronicCond_Heartfailure'] == 1]
df_train_op_Hf.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_op_Hf)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_op_Hf.PotentialFraud)

In [ ]:
df_train_op_Hf_not_fraud = df_train_op_Hf[df_train_op_Hf['PotentialFraud'] == "No"]

In [ ]:
lower_quantile, middle_quantile,upper_quantile = df_train_op_Hf_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_op_Hf['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_op_Hf['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_op_Hf['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_op_Hf['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_op_Hf['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_op_Hf['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_op_Hf['Hf_claim_op_insurance_tier'] = np.select(conditions, values)

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('Hf_claim_op_insurance_tier', hue= 'PotentialFraud', data = df_train_op_Hf)
plt.xticks(size = 12)
plt.xlabel('Hf_claim_op_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_op_Hf.Hf_claim_op_insurance_tier,2,2)

In [ ]:
df_train_op_total['Hf_claim_op_insurance_tier'] = df_train_op_Hf['Hf_claim_op_insurance_tier']
df_train_op_total["Hf_claim_op_insurance_tier"].fillna("None", inplace = True)
df_train_op_total.head()

### Kidney Disease

In [ ]:
# df_train_ip_Alz = df_train_ip_total[(df_train_ip_total['ChronicCond_Alzheimer'] == 1) & (df_train_ip_total['ChronicCond_Heartfailure'] == 2) & (df_train_ip_total['ChronicCond_KidneyDisease'] == 2) & (df_train_ip_total['ChronicCond_Cancer'] == 2) & (df_train_ip_total['ChronicCond_ObstrPulmonary'] == 2) & (df_train_ip_total['ChronicCond_Depression'] == 2) & (df_train_ip_total['ChronicCond_Diabetes'] == 2) & (df_train_ip_total['ChronicCond_IschemicHeart'] == 2) & (df_train_ip_total['ChronicCond_Osteoporasis'] == 2) & (df_train_ip_total['ChronicCond_rheumatoidarthritis'] == 2)& (df_train_ip_total['ChronicCond_stroke'] == 2)]
df_train_op_Kd = df_train_op_total[df_train_op_total['ChronicCond_KidneyDisease'] == 1]
df_train_op_Kd.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_op_Kd)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_op_Kd.PotentialFraud)

In [ ]:
df_train_op_Kd_not_fraud = df_train_op_Kd[df_train_op_Kd['PotentialFraud'] == "No"]

lower_quantile, middle_quantile,upper_quantile = df_train_op_Kd_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_op_Kd['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_op_Kd['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_op_Kd['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_op_Kd['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_op_Kd['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_op_Kd['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_op_Kd['Kd_claim_op_insurance_tier'] = np.select(conditions, values)

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('Kd_claim_op_insurance_tier', hue= 'PotentialFraud', data = df_train_op_Kd)
plt.xticks(size = 12)
plt.xlabel('Kd_claim_op_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_op_Kd.Kd_claim_op_insurance_tier,2,2)

In [ ]:
df_train_op_total['Kd_claim_op_insurance_tier'] = df_train_op_Kd['Kd_claim_op_insurance_tier']
df_train_op_total["Kd_claim_op_insurance_tier"].fillna("None", inplace = True)
df_train_op_total.head()

### Cancer

In [ ]:
df_train_op_Ca = df_train_op_total[df_train_op_total['ChronicCond_Cancer'] == 1]
df_train_op_Ca.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_op_Ca)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_op_Ca.PotentialFraud)

In [ ]:
df_train_op_Ca_not_fraud = df_train_op_Ca[df_train_op_Ca['PotentialFraud'] == "No"]

# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_op_Ca_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_op_Ca['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_op_Ca['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_op_Ca['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_op_Ca['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_op_Ca['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_op_Ca['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_op_Ca['Ca_claim_op_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Ca_claim_op_insurance_tier', hue= 'PotentialFraud', data = df_train_op_Ca)
plt.xticks(size = 12)
plt.xlabel('Ca_claim_op_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_op_Ca.Ca_claim_op_insurance_tier,2,2)

In [ ]:
df_train_op_total['Ca_claim_op_insurance_tier'] = df_train_op_Ca['Ca_claim_op_insurance_tier']
df_train_op_total["Ca_claim_op_insurance_tier"].fillna("None", inplace = True)
df_train_op_total.head()

### ObstrPulmonary

In [ ]:
df_train_op_Obs = df_train_op_total[df_train_op_total['ChronicCond_ObstrPulmonary'] == 1]
df_train_op_Obs.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_op_Obs)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_op_Obs.PotentialFraud)

In [ ]:
df_train_op_Obs_not_fraud = df_train_op_Obs[df_train_op_Obs['PotentialFraud'] == "No"]

# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_op_Obs_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_op_Obs['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_op_Obs['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_op_Obs['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_op_Obs['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_op_Obs['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_op_Obs['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_op_Obs['Obs_claim_op_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Obs_claim_op_insurance_tier', hue= 'PotentialFraud', data = df_train_op_Obs)
plt.xticks(size = 12)
plt.xlabel('Obs_claim_op_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_op_Obs.Obs_claim_op_insurance_tier,2,2)

In [ ]:
df_train_op_total['Obs_claim_op_insurance_tier'] = df_train_op_Obs['Obs_claim_op_insurance_tier']
df_train_op_total["Obs_claim_op_insurance_tier"].fillna("None", inplace = True)
df_train_op_total.head()

### Depression

In [ ]:
df_train_op_Dep = df_train_op_total[df_train_op_total['ChronicCond_Depression'] == 1]
df_train_op_Dep.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_op_Dep)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_op_Dep.PotentialFraud)

In [ ]:
df_train_op_Dep_not_fraud = df_train_op_Dep[df_train_op_Dep['PotentialFraud'] == "No"]

# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_op_Dep_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_op_Dep['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_op_Dep['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_op_Dep['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_op_Dep['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_op_Dep['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_op_Dep['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_op_Dep['Dep_claim_op_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Dep_claim_op_insurance_tier', hue= 'PotentialFraud', data = df_train_op_Dep)
plt.xticks(size = 12)
plt.xlabel('Dep_claim_op_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_op_Dep.Dep_claim_op_insurance_tier,2,2)

In [ ]:
df_train_op_total['Dep_claim_op_insurance_tier'] = df_train_op_Dep['Dep_claim_op_insurance_tier']
df_train_op_total["Dep_claim_op_insurance_tier"].fillna("None", inplace = True)
df_train_op_total.head()

### Diabetes

In [ ]:
df_train_op_Dia = df_train_op_total[df_train_op_total['ChronicCond_Diabetes'] == 1]
df_train_op_Dia.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_op_Dia)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_op_Dia.PotentialFraud)

In [ ]:
df_train_op_Dia_not_fraud = df_train_op_Dia[df_train_op_Dia['PotentialFraud'] == "No"]

# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_op_Dia_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_op_Dia['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_op_Dia['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_op_Dia['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_op_Dia['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_op_Dia['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_op_Dia['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_op_Dia['Dia_claim_op_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Dia_claim_op_insurance_tier', hue= 'PotentialFraud', data = df_train_op_Dia)
plt.xticks(size = 12)
plt.xlabel('Dia_claim_op_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_op_Dia.Dia_claim_op_insurance_tier,2,2)

In [ ]:
df_train_op_total['Dia_claim_op_insurance_tier'] = df_train_op_Dia['Dia_claim_op_insurance_tier']
df_train_op_total["Dia_claim_op_insurance_tier"].fillna("None", inplace = True)
df_train_op_total.head()

### IschemicHeart

In [ ]:
df_train_op_Isc = df_train_op_total[df_train_op_total['ChronicCond_IschemicHeart'] == 1]
df_train_op_Isc.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_op_Isc)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_op_Isc.PotentialFraud)

In [ ]:
df_train_op_Isc_not_fraud = df_train_op_Isc[df_train_op_Isc['PotentialFraud'] == "No"]
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_op_Isc_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_op_Isc['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_op_Isc['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_op_Isc['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_op_Isc['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_op_Isc['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_op_Isc['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_op_Isc['Isc_claim_op_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Isc_claim_op_insurance_tier', hue= 'PotentialFraud', data = df_train_op_Isc)
plt.xticks(size = 12)
plt.xlabel('Isc_claim_op_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_op_Isc.Isc_claim_op_insurance_tier,2,2)

In [ ]:
df_train_op_total['Isc_claim_op_insurance_tier'] = df_train_op_Isc['Isc_claim_op_insurance_tier']
df_train_op_total["Isc_claim_op_insurance_tier"].fillna("None", inplace = True)
df_train_op_total.head()

### Osteoporasis

In [ ]:
df_train_op_Ost = df_train_op_total[df_train_op_total['ChronicCond_Osteoporasis'] == 1]
df_train_op_Ost.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_op_Ost)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_op_Ost.PotentialFraud)

In [ ]:
df_train_op_Ost_not_fraud = df_train_op_Ost[df_train_op_Ost['PotentialFraud'] == "No"]
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_op_Ost_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_op_Ost['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_op_Ost['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_op_Ost['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_op_Ost['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_op_Ost['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_op_Ost['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_op_Ost['Ost_claim_op_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Ost_claim_op_insurance_tier', hue= 'PotentialFraud', data = df_train_op_Ost)
plt.xticks(size = 12)
plt.xlabel('Ost_claim_op_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_op_Ost.Ost_claim_op_insurance_tier,2,2)

In [ ]:
df_train_op_total['Ost_claim_op_insurance_tier'] = df_train_op_Ost['Ost_claim_op_insurance_tier']
df_train_op_total["Ost_claim_op_insurance_tier"].fillna("None", inplace = True)
df_train_op_total.head()

### Rheumatoidarthritis

In [ ]:
df_train_op_Rhe = df_train_op_total[df_train_op_total['ChronicCond_rheumatoidarthritis'] == 1]
df_train_op_Rhe.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_op_Rhe)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_op_Rhe.PotentialFraud)

In [ ]:
df_train_op_Rhe_not_fraud = df_train_op_Rhe[df_train_op_Rhe['PotentialFraud'] == "No"]
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_op_Rhe_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_op_Rhe['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_op_Rhe['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_op_Rhe['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_op_Rhe['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_op_Rhe['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_op_Rhe['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['very_expensive', 'expensive', 'regular', 'inexpensive']
df_train_op_Rhe['Rhe_claim_op_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Rhe_claim_op_insurance_tier', hue= 'PotentialFraud', data = df_train_op_Rhe)
plt.xticks(size = 12)
plt.xlabel('Rhe_claim_op_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_op_Rhe.Rhe_claim_op_insurance_tier,2,2)

In [ ]:
df_train_op_total['Rhe_claim_op_insurance_tier'] = df_train_op_Rhe['Rhe_claim_op_insurance_tier']
df_train_op_total["Rhe_claim_op_insurance_tier"].fillna("None", inplace = True)
df_train_op_total.head()

### Stroke

In [ ]:
df_train_op_Str = df_train_op_total[df_train_op_total['ChronicCond_Depression'] == 1]
df_train_op_Str.shape

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot('PotentialFraud', data = df_train_op_Str)
plt.xticks(size = 12)
plt.xlabel('PotentialFraud', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
without_hue(ax,df_train_op_Str.PotentialFraud)

In [ ]:
df_train_op_Str_not_fraud = df_train_op_Str[df_train_op_Str['PotentialFraud'] == "No"]
# average_insurance_claim = df_train_ip_Alz_not_fraud[df_train_ip_Alz_not_fraud.InscClaimAmtReimbursed != 0].InscClaimAmtReimbursed.mean()
lower_quantile, middle_quantile,upper_quantile = df_train_op_Str_not_fraud.InscClaimAmtReimbursed.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_op_Str['InscClaimAmtReimbursed'] <= lower_quantile),
    (df_train_op_Str['InscClaimAmtReimbursed'] > lower_quantile) & (df_train_op_Str['InscClaimAmtReimbursed'] <= middle_quantile),
    (df_train_op_Str['InscClaimAmtReimbursed'] > middle_quantile) & (df_train_op_Str['InscClaimAmtReimbursed'] <= upper_quantile),
    (df_train_op_Str['InscClaimAmtReimbursed'] > upper_quantile)
    ]

values = ['inexpensive','regular','expensive' ,'very_expensive']
df_train_op_Str['Str_claim_op_insurance_tier'] = np.select(conditions, values)

plt.figure(figsize=(7,5))
ax = sns.countplot('Str_claim_op_insurance_tier', hue= 'PotentialFraud', data = df_train_op_Str)
plt.xticks(size = 12)
plt.xlabel('Str_claim_op_insurance_tier', size = 12)
plt.yticks(size = 12)
plt.ylabel('count', size = 12)
with_hue(ax, df_train_op_Str.Str_claim_op_insurance_tier,2,2)

In [ ]:
df_train_op_total['Str_claim_op_insurance_tier'] = df_train_op_Str['Str_claim_op_insurance_tier']
df_train_op_total["Str_claim_op_insurance_tier"].fillna("None", inplace = True)
df_train_op_total.head()

# Concat Two Dataframe

In [ ]:
# df_train_total=df_train_total.drop('AdmissionDt')
frames = [df_train_ip_total, df_train_op_total]
df_train_total = pd.concat(frames)
df_train_total = df_train_total.drop('AdmissionDt', axis=1)
df_train_total = df_train_total.fillna("None")
df_train_total_cpy = df_train_total.copy()

In [ ]:
df_train_total_cpy.head()

# Check Repetitions between Fraud Providers with the AttendingPhysician

In [ ]:
df_train_total_fraud = df_train_total[df_train_total['PotentialFraud'] == "Yes"]

In [ ]:
df_train_total_fraud['AttendingPhysician'].replace(to_replace=["None"], value=np.nan, inplace=True)

In [ ]:
df_train_total_fraud.head()

In [ ]:
df_train_total_fraud= df_train_total_fraud.groupby(['AttendingPhysician', 'Provider']).size().reset_index(name='counts')

In [ ]:
df_train_total_fraud.counts

In [ ]:
df_train_total_fraud.head()

In [ ]:
lower_quantile, middle_quantile,upper_quantile = df_train_total_fraud.counts.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_total_fraud['counts'] <= lower_quantile),
    (df_train_total_fraud['counts'] > lower_quantile) & (df_train_total_fraud['counts'] <= middle_quantile),
    (df_train_total_fraud['counts'] > middle_quantile) & (df_train_total_fraud['counts'] <= upper_quantile),
    (df_train_total_fraud['counts'] > upper_quantile)
    ]

values = ['normal','suspicious','very_suspicious' ,'most_probably_fraud']
df_train_total_fraud['Physician_Potential_Fraud'] = np.select(conditions, values)

In [ ]:
df_train_total_fraud

In [ ]:
df_train_total.head()

In [ ]:
df_train_total_cpy['Physician_Potential_Fraud'] = df_train_total_fraud['Physician_Potential_Fraud']
df_train_total_cpy["Physician_Potential_Fraud"].fillna("None", inplace = True)
df_train_total_cpy.head()

# Beneficiaries with concerning level of reimbursement

## IPAnnualReimbursementAmt

In [ ]:
max_Ip_annual_reimbursement = df_train_total_cpy[df_train_total_cpy.IPAnnualReimbursementAmt == df_train_total_cpy.IPAnnualReimbursementAmt.max()]

In [ ]:
max_Ip_annual_reimbursement

In [ ]:
max_Ip_annual_reimbursement['PotentialFraud']

In [ ]:
df_bene_fraud = df_train_total_cpy[df_train_total_cpy.BeneID == 'BENE155688']
df_bene_fraud

## OPAnnualReimbursementAmt

In [ ]:
max_Op_annual_reimbursement = df_train_total_cpy[df_train_total_cpy.OPAnnualReimbursementAmt == df_train_total_cpy.OPAnnualReimbursementAmt.max()]
max_Ip_annual_reimbursement

There is no cooling-off period (https://www.hdfcergo.com/blogs/health-insurance/what-is-cooling-off-period-in-health-insurance)

# Check Repetitions between Fraud Providers with the OperatingPhysician

In [ ]:
df_train_total_fraud = df_train_total_cpy[df_train_total_cpy['PotentialFraud'] == "Yes"]
df_train_total_fraud['OperatingPhysician'].replace(to_replace=["None"], value=np.nan, inplace=True)
df_train_total_fraud= df_train_total_fraud.groupby(['OperatingPhysician', 'Provider']).size().reset_index(name='counts')

In [ ]:
df_train_total_fraud.head()

In [ ]:
df_train_total_fraud.counts

In [ ]:
lower_quantile, middle_quantile,upper_quantile = df_train_total_fraud.counts.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_total_fraud['counts'] <= lower_quantile),
    (df_train_total_fraud['counts'] > lower_quantile) & (df_train_total_fraud['counts'] <= middle_quantile),
    (df_train_total_fraud['counts'] > middle_quantile) & (df_train_total_fraud['counts'] <= upper_quantile),
    (df_train_total_fraud['counts'] > upper_quantile)
    ]

values = ['normal','suspicious','very_suspicious' ,'most_probably_fraud']
df_train_total_fraud['Operating_Physician_Potential_Fraud'] = np.select(conditions, values)

In [ ]:
df_train_total_cpy['Operating_Physician_Potential_Fraud'] = df_train_total_fraud['Operating_Physician_Potential_Fraud']
df_train_total_cpy["Operating_Physician_Potential_Fraud"].fillna("None", inplace = True)
df_train_total_cpy.head()

# Check Repetitions between Fraud Providers with the OtherPhysician

In [ ]:
df_train_total_fraud = df_train_total_cpy[df_train_total_cpy['PotentialFraud'] == "Yes"]
df_train_total_fraud['OtherPhysician'].replace(to_replace=["None"], value=np.nan, inplace=True)
df_train_total_fraud= df_train_total_fraud.groupby(['OtherPhysician', 'Provider']).size().reset_index(name='counts')

In [ ]:
df_train_total_fraud.counts

In [ ]:
lower_quantile, middle_quantile,upper_quantile = df_train_total_fraud.counts.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_total_fraud['counts'] <= lower_quantile),
    (df_train_total_fraud['counts'] > lower_quantile) & (df_train_total_fraud['counts'] <= middle_quantile),
    (df_train_total_fraud['counts'] > middle_quantile) & (df_train_total_fraud['counts'] <= upper_quantile),
    (df_train_total_fraud['counts'] > upper_quantile)
    ]

values = ['normal','suspicious','very_suspicious' ,'most_probably_fraud']
df_train_total_fraud['Other_Physician_Potential_Fraud'] = np.select(conditions, values)

In [ ]:
df_train_total_cpy['Other_Physician_Potential_Fraud'] = df_train_total_fraud['Other_Physician_Potential_Fraud']
df_train_total_cpy["Other_Physician_Potential_Fraud"].fillna("None", inplace = True)
df_train_total_cpy.head()

# Check Repetitions between Fraud Providers with the Beneficiaries

In [ ]:
df_train_total_fraud = df_train_total_cpy[df_train_total_cpy['PotentialFraud'] == "Yes"]
df_train_total_fraud= df_train_total_fraud.groupby(['BeneID', 'Provider']).size().reset_index(name='counts')

In [ ]:
df_train_total_fraud.counts

In [ ]:
lower_quantile, middle_quantile,upper_quantile = df_train_total_fraud.counts.quantile([0.25,0.5,0.75])

conditions = [
    (df_train_total_fraud['counts'] <= lower_quantile),
    (df_train_total_fraud['counts'] > lower_quantile) & (df_train_total_fraud['counts'] <= middle_quantile),
    (df_train_total_fraud['counts'] > middle_quantile) & (df_train_total_fraud['counts'] <= upper_quantile),
    (df_train_total_fraud['counts'] > upper_quantile)
    ]

values = ['normal','suspicious','very_suspicious' ,'most_probably_fraud']
df_train_total_fraud['Beneficiaries_Potential_Fraud'] = np.select(conditions, values)

In [ ]:
df_train_total_cpy['Beneficiaries_Potential_Fraud'] = df_train_total_fraud['Beneficiaries_Potential_Fraud']
df_train_total_cpy["Beneficiaries_Potential_Fraud"].fillna("None", inplace = True)
df_train_total_cpy.head()

In [ ]:
df_train_total_cpy.columns

In [ ]:
df_train_total = df_train_total_cpy

# Data Preprocessing

In [ ]:
import sklearn.metrics as metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score,  confusion_matrix,classification_report
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['Age','Alz_claim_ip_insurance_tier',
       'Hf_claim_ip_insurance_tier', 'Kd_claim_insurance_tier',
       'Ca_claim_ip_insurance_tier', 'Obs_claim_ip_insurance_tier',
       'Dep_claim_ip_insurance_tier', 'Dia_claim_ip_insurance_tier',
       'Isc_claim_ip_insurance_tier', 'Ost_claim_ip_insurance_tier',
       'Rhe_claim_ip_insurance_tier', 'Str_claim_ip_insurance_tier',
       'Alz_claim_op_insurance_tier',
       'Hf_claim_op_insurance_tier', 'Kd_claim_op_insurance_tier',
       'Ca_claim_op_insurance_tier', 'Obs_claim_op_insurance_tier',
       'Dep_claim_op_insurance_tier', 'Dia_claim_op_insurance_tier',
       'Isc_claim_op_insurance_tier', 'Ost_claim_op_insurance_tier',
       'Rhe_claim_op_insurance_tier', 'Str_claim_op_insurance_tier',
       'Physician_Potential_Fraud', 'Operating_Physician_Potential_Fraud',
       'Other_Physician_Potential_Fraud', 'Beneficiaries_Potential_Fraud','PotentialFraud']

In [ ]:
le = LabelEncoder()
df_train_total[categorical_cols] = df_train_total[categorical_cols].apply(lambda col: le.fit_transform(col))
numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
df_train_total = df_train_total.select_dtypes(include=numerics)

In [ ]:
df_train_total['PotentialFraud']

In [ ]:
# X = df_train_total.drop(['BeneID','ClaimID','ClaimStartDt','ClaimEndDt','Provider','InscClaimAmtReimbursed','AttendingPhysician','OperatingPhysician','OtherPhysician','DOD','DOB','PotentialFraud'],axis=1)
X = df_train_total.drop(['PotentialFraud'],axis=1)
y = df_train_total['PotentialFraud']

X.head()

In [ ]:
X.shape

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_true = train_test_split(X, y, 
                                                                train_size=0.8, test_size=0.2,
                                                                random_state=0)

In [ ]:
import itertools
def plot_confusion_matrix(cm, classes,normalize=False, title='Confusion matrix', cmap=plt.cm.Reds):

     plt.imshow(cm, interpolation='nearest', cmap=cmap)
     plt.title(title)
     plt.colorbar()
     tick_marks = np.arange(len(classes))
     plt.xticks(tick_marks, classes, rotation=45)
     plt.yticks(tick_marks, classes)
     
     thresh = cm.max() / 2.
     for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
         plt.text(j, i, cm[i, j],
             horizontalalignment="center",
             color="white" if cm[i, j] > thresh else "black")

     plt.tight_layout()
     plt.ylabel('Predicted label')
     plt.xlabel('Actual label')


In [ ]:
model = DecisionTreeClassifier()

model.fit(x_train, y_train)

y_pred = model.predict(x_test)

acc_score_dt = accuracy_score(y_true, y_pred)
f1_scores_dt = f1_score(y_true, y_pred)
recall_scores_dt = recall_score(y_true, y_pred)
precision_scores_dt = precision_score(y_true, y_pred)

conf_mat = confusion_matrix(y_true=y_true, y_pred=y_pred)
conf_mat_plot_labels = ['0', '1']
plot_confusion_matrix(cm=conf_mat, classes=conf_mat_plot_labels, title='Confusion Matrix')

In [ ]:
from tabulate import tabulate
table = [['Model', 'Accuracy', 'F1','Recall','Precision'], ['Decision Tree', acc_score_dt, f1_scores_dt, recall_scores_dt,precision_scores_dt]]
print(tabulate(table, headers='firstrow', tablefmt='fancy_grid'))